In [10]:
# =============================================================================
# [FILE 1] miryang_analysis.py
# 밀양시 교통사고 위험 분석 — 모델링 / XAI / 비교실험 / 시각화 통합본
#
# 입력 : 독립변수_추가_09.11.csv (단일 파일)
# 출력 : Tables (CSV) + Figures (PNG) — 한국정보통신학회 투고 규정 준수
#        (그림 제목 없음, 캡션은 논문 본문에 기재)
#
# 설치 : pip install lightgbm xgboost catboost shap dice-ml
#         pip install scikit-learn pytorch-tabnet
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.linear_model   import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble        import RandomForestRegressor, RandomForestClassifier
from sklearn.neural_network  import MLPRegressor, MLPClassifier
from xgboost                 import XGBRegressor, XGBClassifier
from catboost                import CatBoostRegressor, CatBoostClassifier
from lightgbm                import LGBMRegressor, LGBMClassifier
import shap

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size'  : 11,
    'figure.dpi' : 150,
})
pd.set_option('display.max_columns', 100)

try:
    from pytorch_tabnet.tab_model import TabNetRegressor, TabNetClassifier
    TABNET_OK = True
except (ImportError, OSError):
    TABNET_OK = False
    print("[INFO] TabNet 로드 실패 (DLL 또는 미설치) — 생략하고 계속합니다.")

# DiCE (선택적)
try:
    import dice_ml
    from dice_ml import Dice
    DICE_OK = True
except ImportError:
    DICE_OK = False
    print("[INFO] DiCE 미설치 — 생략됩니다. pip install dice-ml")


# =============================================================================
# 공통 설정
# =============================================================================

PROPOSED  = 'LightGBM (Ours)'   # 제안 모델 레이블 (그래프 강조용)
TEST_SIZE = 0.2
SEED      = 42

# 한국어 → 영어 컬럼 매핑 (CSV 컬럼명에 맞게 추가/수정)
COL_EN = {
    'traffic_weight'                        : 'traffic_risk_index',
    '법규위반_안전운전불이행_건수'          : 'viol_unsafe_driving',
    '법규위반_보행자보호의무위반_건수'      : 'viol_pedestrian_prot',
    '법규위반_신호위반_건수'                : 'viol_signal',
    '법규위반_중앙선침범_건수'              : 'viol_centerline',
    '법규위반_교차로통행방법위반_건수'      : 'viol_intersection_method',
    '법규위반_안전거리미확보_건수'          : 'viol_unsafe_distance',
    '법규위반_과속_건수'                    : 'viol_speeding',
    '법규위반_앞지르기방법위반_건수'        : 'viol_overtaking',
    '법규위반_기타_건수'                    : 'viol_other',
    '법규위반_차로위반_건수'                : 'viol_lane',
    '법규위반_철길건널목통과방법위반_건수'  : 'viol_railroad',
    '노면상태_건조_건수'                    : 'road_dry',
    '노면상태_젖음습기_건수'               : 'road_wet',
    '노면상태_서리빙판_건수'               : 'road_frost',
    '노면상태_적설_건수'                   : 'road_snow',
    '노면상태_기타_건수'                   : 'road_other',
    '기상상태_맑음_건수'                   : 'weather_clear',
    '기상상태_흐림_건수'                   : 'weather_cloudy',
    '기상상태_비_건수'                     : 'weather_rain',
    '기상상태_안개_건수'                   : 'weather_fog',
    '기상상태_눈_건수'                     : 'weather_snow',
    '기상상태_기타_건수'                   : 'weather_other',
    '도로형태_단일로_기타_건수'             : 'road_single_other',
    '도로형태_교차로안_건수'               : 'road_intersection',
    '도로형태_교차로부근_건수'             : 'road_near_intersection',
    '도로형태_기타_건수'                   : 'road_type_other',
    '도로형태_터널안_건수'                 : 'road_tunnel',
    '도로형태_고가도로위_건수'             : 'road_overpass',
    '도로형태_지하도로안_건수'             : 'road_underpass',
    '체육시설_개수'                        : 'sports_facilities',
    '캠핑장_개수'                          : 'campgrounds',
    '인구수'                               : 'population',
    '산업시설_개수'                        : 'industrial_facilities',
    '관광지_개수'                          : 'tourist_spots',
    '도시공원_개수'                        : 'urban_parks',
    '운수업체_개수'                        : 'transport_companies',
    '행사_개수'                            : 'local_events',
    '축산물운반업체_개수'                  : 'livestock_transport',
    '일반운수업체_개수'                    : 'general_transport',
}


# =============================================================================
# STEP 1. 데이터 로드 및 전처리
# =============================================================================
print("\n" + "=" * 60)
print("STEP 1 | Data Loading & Preprocessing")
print("=" * 60)

df_raw    = pd.read_csv('./독립변수_추가_0911.csv')
meta_cols = ['시군구_시군명','사고일시_연도','사고일시_월',
             '시군구_읍면동명','사고일시_분기']
df_meta   = df_raw[[c for c in meta_cols if c in df_raw.columns]].copy()

drop_cols = meta_cols + ['법규위반_불법유턴_건수']
df_model  = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns])
df_model  = df_model.astype(int)

# 컬럼명 영문화
df_model.rename(columns={k:v for k,v in COL_EN.items()
                          if k in df_model.columns}, inplace=True)

TARGET  = 'traffic_risk_index'
y_reg   = df_model[TARGET]
X       = df_model.drop(columns=[TARGET])
X_col   = X.columns.tolist()

# 이진 분류 라벨 (중앙값 기준)
threshold = y_reg.median()
y_cls     = (y_reg >= threshold).astype(int)

print(f"  Rows     : {len(X)}")
print(f"  Features : {X.shape[1]}")
print(f"  Threshold: {threshold:.2f}  |  High-risk: {y_cls.sum()}  Low-risk: {(y_cls==0).sum()}")

# 분할 (회귀 + 분류 동일 인덱스 보장)
X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_cls, test_size=TEST_SIZE, random_state=SEED, stratify=y_cls
)

# 선형/MLP 용 스케일링
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)
X_all_sc = scaler.transform(X)


# =============================================================================
# STEP 2. 제안 모델 학습 (LightGBM 회귀 + 분류)
# =============================================================================
print("\n" + "=" * 60)
print("STEP 2 | Proposed Model Training (LightGBM)")
print("=" * 60)

# ── 회귀 (위험지수 예측, SHAP 추출용) ────────────────────────────────────
reg = LGBMRegressor(n_jobs=-1, random_state=SEED, n_estimators=100, max_depth=2)
reg.fit(X, y_reg)

yp_reg = reg.predict(X_test)
r2_main  = round(r2_score(yr_test, yp_reg), 4)
rmse_main= round(np.sqrt(mean_squared_error(yr_test, yp_reg)), 4)
mae_main = round(mean_absolute_error(yr_test, yp_reg), 4)
print(f"  [Regression]  R²={r2_main}  RMSE={rmse_main}  MAE={mae_main}")

# ── 분류 (고/저위험 판별, Lift·DiCE용) ───────────────────────────────────
clf = LGBMClassifier(n_jobs=-1, random_state=SEED, n_estimators=100, max_depth=2)
clf.fit(X_train, yc_train)

yp_cls  = clf.predict(X_test)
ypr_cls = clf.predict_proba(X_test)[:, 1]
auc_main = round(roc_auc_score(yc_test, ypr_cls), 4)
f1_main  = round(f1_score(yc_test, yp_cls), 4)
print(f"  [Classification]  AUC={auc_main}  F1={f1_main}")
print(classification_report(yc_test, yp_cls, target_names=['Low-Risk','High-Risk']))


# =============================================================================
# STEP 3. SHAP 분석
# =============================================================================
print("\n" + "=" * 60)
print("STEP 3 | SHAP Explainability")
print("=" * 60)

explainer   = shap.Explainer(reg, X)
shap_values = explainer(X)
sv          = explainer.shap_values(X)
df_shap     = pd.DataFrame(sv, columns=X_col)

# 위험인자 / 비교우위인자 추출 (Top-5 per row)
risk_top5 = pd.DataFrame(
    df_shap.apply(lambda r: r.nlargest(5).index.tolist(), axis=1).tolist(),
    columns=[f'risk_factor_{i}' for i in range(1,6)]
)
safe_top5 = pd.DataFrame(
    df_shap.apply(lambda r: r.nsmallest(5).index.tolist(), axis=1).tolist(),
    columns=[f'safe_factor_{i}' for i in range(1,6)]
)
print("  SHAP factor extraction complete.")


# =============================================================================
# STEP 4. Lift Chart + 과도경보 탐지
# =============================================================================
print("\n" + "=" * 60)
print("STEP 4 | Lift Chart & Alert Classification")
print("=" * 60)

def compute_lift(y_true, y_prob, n_bins=10):
    df_l  = pd.DataFrame({'y_true': y_true.values, 'y_prob': y_prob})
    df_l  = df_l.sort_values('y_prob', ascending=False).reset_index(drop=True)
    pos   = y_true.sum(); n = len(df_l)
    rows  = []; cum = 0
    for i in range(n_bins):
        chunk = df_l.iloc[i*(n//n_bins):(i+1)*(n//n_bins)]
        cum  += chunk['y_true'].sum()
        rows.append({
            'Top-N% Selected'    : round((i+1)/n_bins*100),
            'Cumulative Gain (%)': round(cum/pos*100, 2),
            'Lift'               : round(chunk['y_true'].mean()/(pos/n), 3)
        })
    return pd.DataFrame(rows)

df_lift = compute_lift(yc_test, ypr_cls)

# 경보 분류
df_eval = X_test.copy().reset_index(drop=True)
df_eval['actual']    = yc_test.values
df_eval['predicted'] = yp_cls
df_eval['prob']      = ypr_cls.round(4)

true_alert = df_eval[(df_eval['predicted']==1)&(df_eval['actual']==1)].copy()
true_alert['alert_type'] = 'True-Alert'
over_warn  = df_eval[
    (df_eval['predicted']==1) &
    (df_eval['actual']==0) &
    (df_eval['prob'] >= np.percentile(ypr_cls, 70))
].copy()
over_warn['alert_type'] = 'Over-Warning'

print(f"  True Alerts  : {len(true_alert)}")
print(f"  Over-Warnings: {len(over_warn)}")


# =============================================================================
# STEP 5. DiCE 반사실적 설명
# =============================================================================
print("\n" + "=" * 60)
print("STEP 5 | DiCE Counterfactual Explanations")
print("=" * 60)

DICE_AVAILABLE = False
df_policy = pd.DataFrame()

if DICE_OK:
    try:
        df_dice_in = X_train.copy()
        df_dice_in['high_risk'] = yc_train.values
        d   = dice_ml.Data(dataframe=df_dice_in,
                           continuous_features=X_col,
                           outcome_name='high_risk')
        m   = dice_ml.Model(model=clf, backend='sklearn')
        exp = Dice(d, m, method='random')

        hi_idx    = X_test[yp_cls==1].head(5)
        cf_list   = []
        orig_vals = hi_idx.reset_index(drop=True)

        for i, (ridx, row) in enumerate(hi_idx.iterrows()):
            dice_exp = exp.generate_counterfactuals(
                query_instances=row.to_frame().T,
                total_CFs=3, desired_class='opposite', features_to_vary='all'
            )
            cf_df = dice_exp.cf_examples_list[0].final_cfs_df.copy()
            cf_df['instance_id'] = i+1
            cf_df['scenario']    = [f'CF-{j+1}' for j in range(len(cf_df))]
            cf_list.append(cf_df)

        df_dice_all = pd.concat(cf_list, ignore_index=True)

        # 정책 변수 요약
        policy_rows = []
        for i in range(len(orig_vals)):
            subset = df_dice_all[df_dice_all['instance_id']==i+1]
            for _, cf_row in subset.iterrows():
                for col in X_col:
                    delta = cf_row[col] - orig_vals.loc[i, col]
                    if abs(delta) > 0:
                        policy_rows.append({
                            'Feature'         : col,
                            'Original'        : orig_vals.loc[i, col],
                            'Counterfactual'  : cf_row[col],
                            'Delta'           : round(delta, 2),
                            'Direction'       : 'Decrease' if delta<0 else 'Increase',
                            'Scenario'        : cf_row['scenario']
                        })
        df_policy = pd.DataFrame(policy_rows)
        DICE_AVAILABLE = True
        print("  DiCE counterfactuals generated.")
    except Exception as e:
        print(f"  [DiCE Error] {e}")
else:
    print("  [SKIP] dice-ml not installed.")


# =============================================================================
# STEP 6. 모델 비교 실험 (9개 알고리즘)
# =============================================================================
print("\n" + "=" * 60)
print("STEP 6 | Model Comparison (9 Algorithms)")
print("=" * 60)

REG_MODELS = {
    'Linear Reg.'  : (LinearRegression(), False),
    'Ridge Reg.'   : (Ridge(alpha=1.0, random_state=SEED), False),
    'Lasso Reg.'   : (Lasso(alpha=0.1, max_iter=5000, random_state=SEED), False),
    'Random Forest': (RandomForestRegressor(n_estimators=100, max_depth=6,
                                             random_state=SEED, n_jobs=-1), False),
    'XGBoost'      : (XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                    random_state=SEED, verbosity=0, n_jobs=-1), False),
    'CatBoost'     : (CatBoostRegressor(iterations=100, depth=4, learning_rate=0.1,
                                         random_state=SEED, verbose=False), False),
    PROPOSED       : (LGBMRegressor(n_estimators=100, max_depth=2,
                                     random_state=SEED, n_jobs=-1), False),
    'MLP'          : (MLPRegressor(hidden_layer_sizes=(128,64,32),
                                    max_iter=500, random_state=SEED), True),
}

CLS_MODELS = {
    'Logistic Reg.': (LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=6,
                                              random_state=SEED, n_jobs=-1), False),
    'XGBoost'      : (XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                                     random_state=SEED, verbosity=0, n_jobs=-1), False),
    'CatBoost'     : (CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
                                          random_state=SEED, verbose=False), False),
    PROPOSED       : (LGBMClassifier(n_estimators=100, max_depth=2,
                                      random_state=SEED, n_jobs=-1), False),
    'MLP'          : (MLPClassifier(hidden_layer_sizes=(128,64,32),
                                     max_iter=500, random_state=SEED), True),
}

# ── 회귀 비교 ─────────────────────────────────────────────────────────────
reg_rows = []
for name, (model, scaled) in REG_MODELS.items():
    Xtr = X_tr_sc if scaled else X_train.values
    Xte = X_te_sc if scaled else X_test.values
    t0  = time.time()
    model.fit(Xtr, yr_train)
    elapsed = round(time.time()-t0, 2)
    yp = model.predict(Xte)

    kf    = KFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_r2 = []
    Xa    = X_all_sc if scaled else X.values
    for tr, va in kf.split(Xa):
        m2 = type(model)(**model.get_params())
        m2.fit(Xa[tr], y_reg.values[tr])
        cv_r2.append(r2_score(y_reg.values[va], m2.predict(Xa[va])))

    reg_rows.append({
        'Model'        : name,
        'R²'           : round(r2_score(yr_test, yp), 4),
        'RMSE'         : round(np.sqrt(mean_squared_error(yr_test, yp)), 4),
        'MAE'          : round(mean_absolute_error(yr_test, yp), 4),
        'CV R² (mean)' : round(np.mean(cv_r2), 4),
        'CV R² (std)'  : round(np.std(cv_r2), 4),
        'Time (s)'     : elapsed,
    })
    print(f"  {name:<18} R²={reg_rows[-1]['R²']:.4f}  "
          f"RMSE={reg_rows[-1]['RMSE']:.4f}  CV={reg_rows[-1]['CV R² (mean)']:.4f}")

if TABNET_OK:
    t0  = time.time()
    tbn = TabNetRegressor(verbose=0, seed=SEED)
    tbn.fit(X_train.values.astype(np.float32),
            yr_train.values.reshape(-1,1).astype(np.float32),
            eval_set=[(X_test.values.astype(np.float32),
                       yr_test.values.reshape(-1,1).astype(np.float32))],
            patience=20, max_epochs=200, batch_size=256)
    yp = tbn.predict(X_test.values.astype(np.float32)).flatten()
    reg_rows.append({'Model':'TabNet',
                     'R²':round(r2_score(yr_test,yp),4),
                     'RMSE':round(np.sqrt(mean_squared_error(yr_test,yp)),4),
                     'MAE':round(mean_absolute_error(yr_test,yp),4),
                     'CV R² (mean)':'-','CV R² (std)':'-',
                     'Time (s)':round(time.time()-t0,2)})

df_reg_cmp = pd.DataFrame(reg_rows).sort_values('R²', ascending=False)

# ── 분류 비교 ─────────────────────────────────────────────────────────────
cls_rows = []
for name, (model, scaled) in CLS_MODELS.items():
    Xtr = X_tr_sc if scaled else X_train.values
    Xte = X_te_sc if scaled else X_test.values
    t0  = time.time()
    model.fit(Xtr, yc_train)
    elapsed = round(time.time()-t0, 2)
    yp   = model.predict(Xte)
    yprb = model.predict_proba(Xte)[:,1]

    skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_auc  = []
    Xa      = X_all_sc if scaled else X.values
    for tr, va in skf.split(Xa, y_cls.values):
        m2 = type(model)(**model.get_params())
        m2.fit(Xa[tr], y_cls.values[tr])
        cv_auc.append(roc_auc_score(y_cls.values[va],
                                     m2.predict_proba(Xa[va])[:,1]))

    cls_rows.append({
        'Model'         : name,
        'Accuracy'      : round(accuracy_score(yc_test, yp), 4),
        'Precision'     : round(precision_score(yc_test, yp, zero_division=0), 4),
        'Recall'        : round(recall_score(yc_test, yp, zero_division=0), 4),
        'F1-Score'      : round(f1_score(yc_test, yp, zero_division=0), 4),
        'ROC-AUC'       : round(roc_auc_score(yc_test, yprb), 4),
        'CV AUC (mean)' : round(np.mean(cv_auc), 4),
        'CV AUC (std)'  : round(np.std(cv_auc), 4),
        'Time (s)'      : elapsed,
    })
    print(f"  {name:<18} AUC={cls_rows[-1]['ROC-AUC']:.4f}  "
          f"F1={cls_rows[-1]['F1-Score']:.4f}  CV={cls_rows[-1]['CV AUC (mean)']:.4f}")

if TABNET_OK:
    t0  = time.time()
    tbc = TabNetClassifier(verbose=0, seed=SEED)
    tbc.fit(X_train.values.astype(np.float32), yc_train.values,
            eval_set=[(X_test.values.astype(np.float32), yc_test.values)],
            patience=20, max_epochs=200, batch_size=256)
    yp   = tbc.predict(X_test.values.astype(np.float32))
    yprb = tbc.predict_proba(X_test.values.astype(np.float32))[:,1]
    cls_rows.append({'Model':'TabNet',
                     'Accuracy':round(accuracy_score(yc_test,yp),4),
                     'Precision':round(precision_score(yc_test,yp,zero_division=0),4),
                     'Recall':round(recall_score(yc_test,yp,zero_division=0),4),
                     'F1-Score':round(f1_score(yc_test,yp,zero_division=0),4),
                     'ROC-AUC':round(roc_auc_score(yc_test,yprb),4),
                     'CV AUC (mean)':'-','CV AUC (std)':'-',
                     'Time (s)':round(time.time()-t0,2)})

df_cls_cmp = pd.DataFrame(cls_rows).sort_values('ROC-AUC', ascending=False)


# =============================================================================
# STEP 7. Tables 저장
# =============================================================================
print("\n" + "=" * 60)
print("STEP 7 | Saving Tables")
print("=" * 60)

# Table I: 회귀 모델 비교
df_reg_cmp.to_csv('./TableI_Regression_Comparison.csv',
                   index=False, encoding='utf-8-sig')
print("  [Saved] TableI_Regression_Comparison.csv")

# Table II: 분류 모델 비교
df_cls_cmp.to_csv('./TableII_Classification_Comparison.csv',
                   index=False, encoding='utf-8-sig')
print("  [Saved] TableII_Classification_Comparison.csv")

# Table III: Lift Chart 수치
df_lift.to_csv('./TableIII_Lift_Chart.csv', index=False, encoding='utf-8-sig')
print("  [Saved] TableIII_Lift_Chart.csv")

# Table IV: 경보 분류
pd.concat([true_alert, over_warn]).sort_values('prob', ascending=False)\
  .to_csv('./TableIV_Alert_Classification.csv', index=False, encoding='utf-8-sig')
print("  [Saved] TableIV_Alert_Classification.csv")

# Table V: DiCE 정책 변수
if DICE_AVAILABLE:
    df_policy.to_csv('./TableV_DiCE_Policy.csv', index=False, encoding='utf-8-sig')
    print("  [Saved] TableV_DiCE_Policy.csv")

# Table VI: LLM 통합 데이터셋 (chatbot RAG 입력)
df_llm = df_meta.copy().reset_index(drop=True)
df_llm['predicted_risk_index'] = reg.predict(X).round(2)
df_llm['high_risk_flag']       = clf.predict(X)
df_llm['high_risk_prob']       = clf.predict_proba(X)[:,1].round(4)
df_llm['actual_risk_index']    = y_reg.values
df_llm['alert_type']           = df_llm['high_risk_flag'].map(
                                    {1:'High-Risk Warning', 0:'Low-Risk'})
df_llm = pd.concat([df_llm, risk_top5, safe_top5], axis=1)
df_llm.to_csv('./TableVI_LLM_Dataset.csv', index=False, encoding='utf-8-sig')
print("  [Saved] TableVI_LLM_Dataset.csv")


# =============================================================================
# STEP 8. Figures 생성 (제목 없음 — JKIICE 규정)
# =============================================================================
print("\n" + "=" * 60)
print("STEP 8 | Generating Figures")
print("=" * 60)

# ── Fig 1: Confusion Matrix ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5,4))
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(yc_test, yp_cls),
                               display_labels=['Low-Risk','High-Risk'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('')
plt.tight_layout()
plt.savefig('./Fig1_Confusion_Matrix.png', dpi=150, bbox_inches='tight')
plt.close(); print("  [Saved] Fig1_Confusion_Matrix.png")

# ── Fig 2: SHAP Summary Bar ───────────────────────────────────────────────
mean_shap = np.abs(df_shap).mean().sort_values(ascending=False).head(15)
fig, ax   = plt.subplots(figsize=(8,6))
ax.barh(mean_shap.index[::-1], mean_shap.values[::-1],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('./Fig2_SHAP_Bar.png', dpi=150, bbox_inches='tight')
plt.close(); print("  [Saved] Fig2_SHAP_Bar.png")

# ── Fig 3: SHAP Beeswarm ─────────────────────────────────────────────────
fig = plt.figure(figsize=(9,6))
shap.summary_plot(sv, X, plot_type='dot', max_display=15, show=False)
plt.title('')
plt.tight_layout()
plt.savefig('./Fig3_SHAP_Beeswarm.png', dpi=150, bbox_inches='tight')
plt.close(); print("  [Saved] Fig3_SHAP_Beeswarm.png")

# ── Fig 4: Lift Chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].plot(df_lift['Top-N% Selected'], df_lift['Cumulative Gain (%)'],
             marker='o', color='steelblue', lw=2, label='Model')
axes[0].plot([0,100],[0,100],'k--', lw=1.5, label='Random')
axes[0].fill_between(df_lift['Top-N% Selected'],
                     df_lift['Cumulative Gain (%)'],
                     df_lift['Top-N% Selected'], alpha=0.1, color='steelblue')
axes[0].set_xlabel('Top-N% Selected'); axes[0].set_ylabel('Cumulative Gain (%)')
axes[0].set_title('(a) Cumulative Gain Curve')
axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].set_xlim(0,100); axes[0].set_ylim(0,105)

bar_colors = ['#d73027' if v>=1.5 else '#4575b4' if v>=1.0 else '#fdae61'
              for v in df_lift['Lift']]
axes[1].bar(df_lift['Top-N% Selected'].astype(int).astype(str),
            df_lift['Lift'], color=bar_colors, edgecolor='white', width=0.7)
axes[1].axhline(1.0, color='black', linestyle='--', lw=1.5, label='Lift=1')
axes[1].axhline(1.5, color='red',   linestyle=':',  lw=1,   label='Lift=1.5')
axes[1].set_xlabel('Top-N% Decile'); axes[1].set_ylabel('Lift Value')
axes[1].set_title('(b) Lift by Decile')
axes[1].legend(fontsize=9); axes[1].grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('./Fig4_Lift_Chart.png', dpi=150, bbox_inches='tight')
plt.close(); print("  [Saved] Fig4_Lift_Chart.png")

# ── Fig 5: DiCE Policy Variables ─────────────────────────────────────────
if DICE_AVAILABLE:
    top_dec = df_policy[df_policy['Direction']=='Decrease']['Feature']\
                        .value_counts().head(10)
    fig, ax = plt.subplots(figsize=(8,5))
    ax.barh(top_dec.index[::-1], top_dec.values[::-1],
            color='tomato', edgecolor='white')
    ax.set_xlabel('Frequency across Counterfactual Scenarios')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('./Fig5_DiCE_Policy.png', dpi=150, bbox_inches='tight')
    plt.close(); print("  [Saved] Fig5_DiCE_Policy.png")

# ── Fig 6: Analysis Pipeline ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 3.2))
ax.axis('off')
steps  = ['Raw Data\n(Single CSV)','Feature\nEngineering',
          'LGBM\nRegressor','LGBM\nClassifier',
          'SHAP\n(Why risky?)','Lift Chart\n(Over-warning?)',
          'DiCE\n(What to change?)','LLM Chatbot\n(Policy Advice)']
colors = ['#bdbdbd','#b3cde3','#6497b1','#6497b1',
          '#74c476','#fd8d3c','#d62728','#9467bd']
n = len(steps); margin=0.04; span=1-2*margin; gap=span/(n-1); bw,bh=0.09,0.55
for i,(txt,col) in enumerate(zip(steps,colors)):
    x = margin + i*gap
    ax.add_patch(mpatches.FancyBboxPatch(
        (x-bw/2,0.20),bw,bh,boxstyle='round,pad=0.02',
        facecolor=col,edgecolor='white',linewidth=1.5,
        transform=ax.transAxes,clip_on=False))
    ax.text(x,0.475,txt,ha='center',va='center',fontsize=8.5,fontweight='bold',
            color='black' if col=='#bdbdbd' else 'white',transform=ax.transAxes)
    if i<n-1:
        ax.annotate('',xy=(margin+(i+1)*gap-bw/2-0.005,0.475),
                    xytext=(x+bw/2+0.005,0.475),
                    xycoords='axes fraction',textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->',color='#444',lw=1.5))
plt.tight_layout()
plt.savefig('./Fig6_Pipeline.png', dpi=150, bbox_inches='tight')
plt.close(); print("  [Saved] Fig6_Pipeline.png")

# ── Fig 7: Regression Comparison Bar ─────────────────────────────────────
df_rp = df_reg_cmp[df_reg_cmp['CV R² (mean)']!='-'].copy()
fig, axes = plt.subplots(1,3,figsize=(14,5))
for ax, met, better in zip(axes,['R²','RMSE','MAE'],['higher','lower','lower']):
    vals = df_rp.set_index('Model')[met].astype(float)
    cols = ['#d62728' if m==PROPOSED else '#4878cf' for m in vals.index]
    ax.bar(range(len(vals)),vals.values,color=cols,edgecolor='white',width=0.6)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index,rotation=40,ha='right',fontsize=9)
    ax.set_ylabel(met); ax.grid(axis='y',linestyle='--',alpha=0.4)
    bi = int(vals.argmax() if better=='higher' else vals.argmin())
    ax.patches[bi].set_edgecolor('gold'); ax.patches[bi].set_linewidth(2.5)
fig.legend(handles=[mpatches.Patch(facecolor='#d62728',label='Proposed'),
                    mpatches.Patch(facecolor='#4878cf',label='Baseline'),
                    mpatches.Patch(facecolor='white',edgecolor='gold',
                                   linewidth=2.5,label='Best')],
           loc='upper center',ncol=3,fontsize=9,bbox_to_anchor=(0.5,1.02))
plt.tight_layout()
plt.savefig('./Fig7_Regression_Comparison.png',dpi=150,bbox_inches='tight')
plt.close(); print("  [Saved] Fig7_Regression_Comparison.png")

# ── Fig 8: Classification AUC + F1 Grouped Bar ───────────────────────────
df_cp   = df_cls_cmp[df_cls_cmp['CV AUC (mean)']!='-'].copy()
models_c= df_cp['Model'].tolist()
x       = np.arange(len(models_c)); w=0.35
fig, ax = plt.subplots(figsize=(12,5))
b1 = ax.bar(x-w/2,df_cp['ROC-AUC'].astype(float),w,
            color=['#d62728' if m==PROPOSED else '#4878cf' for m in models_c],
            edgecolor='white',label='ROC-AUC')
b2 = ax.bar(x+w/2,df_cp['F1-Score'].astype(float),w,
            color=['#ff7f0e' if m==PROPOSED else '#9ecae1' for m in models_c],
            edgecolor='white',label='F1-Score')
for bar in list(b1)+list(b2):
    h=bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2,h+0.01,f'{h:.3f}',
            ha='center',va='bottom',fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(models_c,rotation=35,ha='right',fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0,1.12)
ax.grid(axis='y',linestyle='--',alpha=0.4)
ax.legend(handles=[mpatches.Patch(facecolor='#d62728',label='AUC (Proposed)'),
                   mpatches.Patch(facecolor='#4878cf',label='AUC (Baseline)'),
                   mpatches.Patch(facecolor='#ff7f0e',label='F1 (Proposed)'),
                   mpatches.Patch(facecolor='#9ecae1',label='F1 (Baseline)')],
          fontsize=9,ncol=2)
plt.tight_layout()
plt.savefig('./Fig8_Classification_Comparison.png',dpi=150,bbox_inches='tight')
plt.close(); print("  [Saved] Fig8_Classification_Comparison.png")

# ── Fig 9: CV AUC Error Bar ───────────────────────────────────────────────
df_cv = df_cp.sort_values('CV AUC (mean)',ascending=True)
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(df_cv['Model'],df_cv['CV AUC (mean)'].astype(float),
        xerr=df_cv['CV AUC (std)'].astype(float),
        color=['#d62728' if m==PROPOSED else '#4878cf' for m in df_cv['Model']],
        edgecolor='white',capsize=4,error_kw={'elinewidth':1.5,'ecolor':'#333'})
ax.set_xlabel('5-Fold CV ROC-AUC (mean ± std)')
ax.axvline(0.9,color='gray',linestyle=':',lw=1)
ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_xlim(0.5,1.05)
ax.legend(handles=[mpatches.Patch(facecolor='#d62728',label='Proposed'),
                   mpatches.Patch(facecolor='#4878cf',label='Baseline')],fontsize=9)
plt.tight_layout()
plt.savefig('./Fig9_CV_AUC_ErrorBar.png',dpi=150,bbox_inches='tight')
plt.close(); print("  [Saved] Fig9_CV_AUC_ErrorBar.png")

# ── Fig 10: Radar Chart (Top-5 classification models) ────────────────────
top5      = df_cls_cmp.head(5)['Model'].tolist()
r_metrics = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
df_radar  = df_cls_cmp[df_cls_cmp['Model'].isin(top5)]\
              .set_index('Model')[r_metrics].astype(float)
N = len(r_metrics)
angles = np.linspace(0,2*np.pi,N,endpoint=False).tolist(); angles+=angles[:1]
fig, ax = plt.subplots(figsize=(7,7),subplot_kw=dict(polar=True))
palette = ['#d62728','#4878cf','#2ca02c','#ff7f0e','#9467bd']
for (model,row),color in zip(df_radar.iterrows(),palette):
    vals = row.tolist()+row.tolist()[:1]
    ax.plot(angles,vals,color=color,lw=2.5 if model==PROPOSED else 1.5,
            linestyle='-' if model==PROPOSED else '--',label=model)
    ax.fill(angles,vals,color=color,alpha=0.05)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(r_metrics,fontsize=10)
ax.set_ylim(0,1); ax.set_yticks([0.6,0.7,0.8,0.9,1.0])
ax.grid(color='gray',linestyle='--',lw=0.5,alpha=0.5)
ax.legend(loc='upper right',bbox_to_anchor=(1.35,1.15),fontsize=9)
plt.tight_layout()
plt.savefig('./Fig10_Radar_Chart.png',dpi=150,bbox_inches='tight')
plt.close(); print("  [Saved] Fig10_Radar_Chart.png")


# =============================================================================
# 최종 요약
# =============================================================================
print("\n" + "=" * 60)
print("COMPLETE — Output Summary")
print("=" * 60)
print(f"  Best Regression    : {df_reg_cmp.iloc[0]['Model']} "
      f"(R²={df_reg_cmp.iloc[0]['R²']})")
print(f"  Best Classification: {df_cls_cmp.iloc[0]['Model']} "
      f"(AUC={df_cls_cmp.iloc[0]['ROC-AUC']})")
print(f"  DiCE available     : {DICE_AVAILABLE}")
print()
print("  Tables : TableI ~ TableVI (.csv)")
print("  Figures: Fig1 ~ Fig10 (.png)")
print("=" * 60)

[INFO] TabNet 로드 실패 (DLL 또는 미설치) — 생략하고 계속합니다.

STEP 1 | Data Loading & Preprocessing
  Rows     : 447
  Features : 33
  Threshold: 35.00  |  High-risk: 229  Low-risk: 218

STEP 2 | Proposed Model Training (LightGBM)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000684 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 108
[LightGBM] [Info] Number of data points in the train set: 447, number of used features: 19
[LightGBM] [Info] Start training from score 47.022371
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Lig

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.56it/s]


  DiCE counterfactuals generated.

STEP 6 | Model Comparison (9 Algorithms)
  Linear Reg.        R²=-63955873226884595712.0000  RMSE=333229122102.3132  CV=-43338311338921715302400.0000
  Ridge Reg.         R²=0.8665  RMSE=15.2260  CV=0.8533
  Lasso Reg.         R²=0.8789  RMSE=14.4987  CV=0.8595
  Random Forest      R²=0.8177  RMSE=17.7890  CV=0.8210
  XGBoost            R²=0.8205  RMSE=17.6525  CV=0.8277
  CatBoost           R²=0.8113  RMSE=18.1003  CV=0.8316
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000256 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 100
[LightGBM] [Info] Number of data points in the train set: 357, number of used features: 18
[LightGBM] [Info] Start training from score 47.098039
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p